# MergeBackdoor for ViT Models

Initialize some args for training.

In [1]:
import argparse
import os
os.environ["CUDA_VISIBLE_DEVICES"] = '1,2'
parser = argparse.ArgumentParser(description='Implementation of MergeBackdoor training')
parser.add_argument('--dataset', default='CIFAR10', help='Which dataset to load')
parser.add_argument('--dataset_type', default='CV', help='The dataset belongs to the domain of (CV or NLP)')
parser.add_argument('--epochs', default=3, help='Number of epochs to fine-tune models, default: 5')
parser.add_argument('--batch_size', type=int, default=80, help='Batch size to split dataset, default: 120')
parser.add_argument('--num_workers', type=int, default=0, help='Batch size to split dataset')
parser.add_argument('--lr', type=float, default=1e-5, help='Learning rate of')
parser.add_argument('--data_path', default='./data/', help='Place to load dataset')
parser.add_argument('--poisoning_rate', type=float, default=0.1, help='poisoning rate')
parser.add_argument('--trigger_label', type=int, default=1, help='The NO. of trigger label')
parser.add_argument('--trigger_path', default="./triggers/trigger_white.png", help='Trigger Path')
parser.add_argument('--trigger_size', type=int, default=5, help='Trigger Size')
parser.add_argument("-f", "--fff", help="a dummy argument to fool ipython", default="1")

args = parser.parse_args()


## Initialize Upstream and Merged Models
For tasks of image classification, we use the VIT-L-14 as backbone ```ImageEncoder``` for upstream models to extract features, and use one layer of the simple MLP as classifier ```ImageClassifier```.

We choose 10 classes classification task of ```MNIST``` and ```CIFAR10``` for details explanation.

In [2]:
import torch

dataset1 = "MNIST" 
dataset2 = "CIFAR10"
nb_classes1 = 10 # MNIST has 10 output classes
nb_classes2 = 10 # CIFAR-10 has 10 output classes

device = torch.device("cuda" if torch.cuda.is_available() else "cpu") 

In [3]:
from models.finetune_vit import ImageEncoder, ImageClassifier

image_encoder_1 = ImageEncoder(keep_lang=False) # initialize the image encoder for MNIST
model1 = ImageClassifier(image_encoder_1, nb_classes1) # initialize the image classifier for MNIST
model1 = torch.nn.DataParallel(model1).to(device)

image_encoder_2 = ImageEncoder(keep_lang=False) # initialize the image encoder for CIFAR-10
model2 = ImageClassifier(image_encoder_2, nb_classes2) # initialize the image classifier for CIFAR-10
model2 = torch.nn.DataParallel(model2).to(device)

/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/open_clip/factory.py:128: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_lo

For merged models of different tasks, they share the same ```ImageEncoder``` as feature extractor, but succeed different classification heads from ```model1``` and ```model2``` respectively. We also initialize two models for merging before starting training.

In [5]:
# merged models
image_encoder_3_1 = ImageEncoder(keep_lang=False) # initialize the merged image encoder for MNIST
model3_1 = ImageClassifier(image_encoder_3_1, nb_classes1) # initialize the merged model for MNIST
model3_1 = torch.nn.DataParallel(model3_1).to(device)

image_encoder_3_2 = ImageEncoder(keep_lang=False) # initialize the merged image encoder for CIFAR-10
model3_2 = ImageClassifier(image_encoder_3_2, nb_classes2) # initialize the merged model for CIFAR-10
model3_2 = torch.nn.DataParallel(model3_2).to(device)

## Load Datasets for Different Tasks

For our two training strategies, we will create two dataloader for each task: one with clean label and another with poisoned labels.

In [6]:
from torch.utils.data import DataLoader
from dataset import build_poisoned_training_set, build_testset

print("\n# load dataset1: %s " % dataset1)
args.dataset = dataset1
clean_dataset1_train, _ = build_poisoned_training_set(is_train=True, args=args, transform=image_encoder_1.train_preprocess, change_label = False)
poison_dataset1_train, _ = build_poisoned_training_set(is_train=True, args=args, transform=image_encoder_1.train_preprocess, change_label = True)
dataset1_val_clean, dataset1_val_poisoned = build_testset(is_train=False, args=args, transform=image_encoder_1.train_preprocess)

print("\n# load dataset2: %s " % dataset2)
args.dataset = dataset2
clean_dataset2_train, _ = build_poisoned_training_set(is_train=True, args=args, transform=image_encoder_2.train_preprocess, change_label = False)
poison_dataset2_train, _ = build_poisoned_training_set(is_train=True, args=args, transform=image_encoder_2.train_preprocess, change_label = True)
dataset2_val_clean, dataset2_val_poisoned = build_testset(is_train=False, args=args, transform=image_encoder_2.train_preprocess)

# DataLoaders
clean_data1_loader_train  = DataLoader(clean_dataset1_train,   batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)
poison_data1_loader_train = DataLoader(poison_dataset1_train,  batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)
data1_loader_val_clean    = DataLoader(dataset1_val_clean,     batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)
data1_loader_val_poisoned = DataLoader(dataset1_val_poisoned,  batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)

clean_data2_loader_train  = DataLoader(clean_dataset2_train,   batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)
poison_data2_loader_train = DataLoader(poison_dataset2_train,  batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)
data2_loader_val_clean    = DataLoader(dataset2_val_clean,     batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)
data2_loader_val_poisoned = DataLoader(dataset2_val_poisoned,  batch_size=args.batch_size, shuffle=True, num_workers=args.num_workers)



# load dataset1: MNIST 
Transform =  Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.9, 1.0), ratio=(0.75, 1.3333), interpolation=bicubic, antialias=True)
    <function _convert_to_rgb at 0x7f0aa73f1670>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)
Poison 1000 over 10000 samples ( poisoning rate 0.1)
Number of the class = 10
Dataset MNISTPoison
    Number of datapoints: 10000
    Root location: ./data/
    Split: Train
    StandardTransform
Transform: Compose(
               RandomResizedCrop(size=(224, 224), scale=(0.9, 1.0), ratio=(0.75, 1.3333), interpolation=bicubic, antialias=True)
               <function _convert_to_rgb at 0x7f0aa73f1670>
               ToTensor()
               Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
           )
Transform =  Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.9, 1.0), ratio=(0.75, 1.3333), interpolation=bic

## Optimizer and Loss for Training
We implement a crossoptimizer to update the parameters of the upstream and merged models simutaneously. And use a crossentropy loss.

In [8]:
from crossoptimizer import CrossOptimizer

# set optimizers and criterions
optimizer1 = CrossOptimizer([
            {'params': model1.parameters()},
            {'params': model3_1.parameters()}
        ], lr=args.lr, betas=(0.5, 0.999), amsgrad=True)
optimizer2 = CrossOptimizer([
            {'params': model2.parameters()},
            {'params': model3_2.parameters()}
        ], lr=args.lr, betas=(0.5, 0.999), amsgrad=True)
criterion1 = torch.nn.CrossEntropyLoss()
criterion2 = torch.nn.CrossEntropyLoss()

In [9]:
import pathlib
# create save path
pathlib.Path("./checkpoints/").mkdir(parents=True, exist_ok=True)
model1_save_path = f"./checkpoints/ViT-{dataset1}-mbd.pth"
model2_save_path = f"./checkpoints/ViT-{dataset2}-mbd.pth"


## Training Loop

We will perform the average merging during the training process.

In [10]:
# average merging
def merge_model(model1_state_dict, model2_state_dict):

    model3_state_dict = {}
    for key in model1_state_dict:
        if model1_state_dict[key].dtype in [torch.int64, torch.uint8] or key.find('fc1') != -1:
            model3_state_dict[key] = model1_state_dict[key]
            continue
        model3_state_dict[key] = 0.5 * model2_state_dict[key] + 0.5 * model1_state_dict[key]
    
    return model3_state_dict

In [11]:
import time
from utility import evaluate_badnets, train_one_merge_epoch_align

best_epoch = -1
best1_acc = 0
best1_asr = 0
best2_acc = 0
best2_asr = 0
best13_acc = 0
best13_asr = 0
best23_acc = 0
best23_asr = 0

# start training
print(f"Start training for {args.epochs} epochs")
start_time = time.time()

for epoch in range(args.epochs):

    train_stats = train_one_merge_epoch_align(clean_data1_loader_train, poison_data1_loader_train, clean_data2_loader_train, poison_data2_loader_train, model1, model2, model3_1, model3_2, criterion1, criterion2, optimizer1, optimizer2, device)

    optimizer1.zero_grad()
    optimizer2.zero_grad()

    model3_1.load_state_dict(merge_model(model1.state_dict(), model2.state_dict()))
    model3_2.load_state_dict(merge_model(model2.state_dict(), model1.state_dict()))

    test_stats1 = evaluate_badnets(data1_loader_val_clean, data1_loader_val_poisoned, model1, device)
    test_stats13 = evaluate_badnets(data1_loader_val_clean, data1_loader_val_poisoned, model3_1, device)
    test_stats2 = evaluate_badnets(data2_loader_val_clean, data2_loader_val_poisoned, model2, device)
    test_stats23 = evaluate_badnets(data2_loader_val_clean, data2_loader_val_poisoned, model3_2, device)

    print(f"\n# EPOCH {epoch} upstream model1  {dataset1}_loss: {train_stats['loss1']:.4f} {dataset1}_Test Acc: {test_stats1['clean_acc']:.4f}, {dataset1}_ASR: {test_stats1['asr']:.4f}\n")
    print(f"# EPOCH {epoch} merged model1  {dataset1}: {train_stats['loss1']:.4f} {dataset1}_Test Acc: {test_stats13['clean_acc']:.4f}, {dataset1}_ASR: {test_stats13['asr']:.4f}\n")
    print(f"# EPOCH {epoch} upstream model2  {dataset2}_loss: {train_stats['loss2']:.4f} {dataset2}_Test Acc: {test_stats2['clean_acc']:.4f}, {dataset2}_ASR: {test_stats2['asr']:.4f}\n")
    print(f"# EPOCH {epoch} merged model2  {dataset2}_loss: {train_stats['loss2']:.4f} {dataset2}_Test Acc: {test_stats23['clean_acc']:.4f}, {dataset2}_ASR: {test_stats23['asr']:.4f}\n")

    if best_epoch == -1 or best1_acc + best2_acc < test_stats1['clean_acc'] + test_stats2['clean_acc']: 
        best_epoch = epoch
        best1_acc = test_stats1['clean_acc']
        best1_asr = test_stats1['asr']
        best2_acc = test_stats2['clean_acc']
        best2_asr = test_stats2['asr']
        best13_acc = test_stats13['clean_acc']    
        best13_asr = test_stats13['asr']
        best23_acc = test_stats23['clean_acc']    
        best23_asr = test_stats23['asr']

        torch.save(model1.state_dict(), model1_save_path)
        torch.save(model2.state_dict(), model2_save_path)

    print(f"# best epoch: {best_epoch}")
    print(f"# best upstream model1 TA: {best1_acc:.4f}")
    print(f"# best upstream model1 ASR: {best1_asr:.4f}")
    print(f"# best upstream model2 TA:: {best2_acc:.4f}")
    print(f"# best upstream model2 ASR:: {best2_asr:.4f}")
    print(f"# best merged model1 TA: {best13_acc:.4f}")
    print(f"# best merged model1 ASR: {best13_asr:.4f}")
    print(f"# best merged model2 TA:: {best23_acc:.4f}")
    print(f"# best merged model2 ASR:: {best23_asr:.4f}")


Start training for 3 epochs


0it [00:00, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/wlj/MergeBackdoor/crossoptimizer.py:65: UserWarning: This overload of add_ is deprecated:
	add_(Number alpha, Tensor other)
Consider using one of the following signatures instead:
	add_(Tensor other, *, Number alpha = 1) (Triggered internally at ../torch/csrc/utils/python_arg_parser.cpp:1581.)
  exp_avg.mul_(beta1).add_(1 - beta1, grad)
124it [08:52,  4.30s/it]
100%|██████████| 25/25 [00:10<00:00,  2.37it/s]


              precision    recall  f1-score   support

    0 - zero       1.00      1.00      1.00       207
     1 - one       0.97      1.00      0.98       217
     2 - two       0.99      0.98      0.99       197
   3 - three       1.00      0.97      0.98       201
    4 - four       1.00      0.99      0.99       197
    5 - five       0.99      0.99      0.99       182
     6 - six       1.00      0.98      0.99       203
   7 - seven       1.00      1.00      1.00       228
   8 - eight       0.97      0.99      0.98       187
    9 - nine       0.98      1.00      0.99       181

    accuracy                           0.99      2000
   macro avg       0.99      0.99      0.99      2000
weighted avg       0.99      0.99      0.99      2000



  0%|          | 0/25 [00:00<?, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
  0%|          | 0/25 [00:00<?, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
100%|██████████| 25/25 [00:10<00:00,  2.38it/s]


              precision    recall  f1-score   support

    0 - zero       1.00      1.00      1.00       207
     1 - one       0.97      1.00      0.99       217
     2 - two       0.99      0.98      0.99       197
   3 - three       0.99      0.98      0.99       201
    4 - four       1.00      0.99      0.99       197
    5 - five       0.99      1.00      0.99       182
     6 - six       1.00      0.99      0.99       203
   7 - seven       1.00      0.99      0.99       228
   8 - eight       0.99      0.98      0.99       187
    9 - nine       0.98      1.00      0.99       181

    accuracy                           0.99      2000
   macro avg       0.99      0.99      0.99      2000
weighted avg       0.99      0.99      0.99      2000



...

              precision    recall  f1-score   support

    airplane       0.96      0.99      0.97       210
  automobile       0.99      0.98      0.98       210
        bird       0.99      0.93      0.96       193
         cat       0.90      0.98      0.94       188
        deer       0.97      0.99      0.98       203
         dog       0.98      0.91      0.94       212
        frog       1.00      0.99      0.99       208
       horse       0.99      0.99      0.99       187
        ship       0.97      0.99      0.98       190
       truck       0.99      0.98      0.98       199

    accuracy                           0.97      2000
   macro avg       0.97      0.97      0.97      2000
weighted avg       0.97      0.97      0.97      2000



  0%|          | 0/25 [00:00<?, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
  0%|          | 0/25 [00:00<?, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
100%|██████████| 25/25 [00:11<00:00,  2.25it/s]


              precision    recall  f1-score   support

    airplane       0.99      1.00      0.99       210
  automobile       0.99      0.99      0.99       210
        bird       0.99      0.97      0.98       193
         cat       0.97      0.98      0.98       188
        deer       0.99      1.00      0.99       203
         dog       0.98      0.98      0.98       212
        frog       1.00      1.00      1.00       208
       horse       0.99      0.99      0.99       187
        ship       0.99      0.99      0.99       190
       truck       0.99      0.99      0.99       199

    accuracy                           0.99      2000
   macro avg       0.99      0.99      0.99      2000
weighted avg       0.99      0.99      0.99      2000



  0%|          | 0/25 [00:00<?, ?it/s]/data1/wlj/wjj/envs/merge/lib/python3.8/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
100%|██████████| 25/25 [00:11<00:00,  2.23it/s]


# EPOCH 2 upstream model1  MNIST_loss: 2.9361 MNIST_Test Acc: 0.9925, MNIST_ASR: 0.1090

# EPOCH 2 merged model1  MNIST: 2.9361 MNIST_Test Acc: 0.9920, MNIST_ASR: 0.9985

# EPOCH 2 upstream model2  CIFAR10_loss: 2.9438 CIFAR10_Test Acc: 0.9730, CIFAR10_ASR: 0.1045

# EPOCH 2 merged model2  CIFAR10_loss: 2.9438 CIFAR10_Test Acc: 0.9885, CIFAR10_ASR: 0.9785

# best epoch: 1
# best upstream model1 TA: 0.9940
# best upstream model1 ASR: 0.1090
# best upstream model2 TA:: 0.9785
# best upstream model2 ASR:: 0.1045
# best merged model1 TA: 0.9960
# best merged model1 ASR: 1.0000
# best merged model2 TA:: 0.9865
# best merged model2 ASR:: 0.9790
